# 5. Longest Palindromic Substring

**Difficulty:** Medium  
**Topics:** Two Pointers, String, Dynamic Programming, Manacher  
**Language:** Python 3  
**Problem:** [LeetCode — Longest Palindromic Substring](https://leetcode.com/problems/longest-palindromic-substring/)

Return a longest contiguous part of a string that reads the same forward and backward. Begin by checking every substring, then remove repeated full checks by expanding outward from possible centres. Each approach includes reasoning, a short walkthrough, correctness, complexity, and complete LeetCode-compatible code.

Run the notebook from top to bottom; only Python's standard library is needed. Edit the JSON string in **Try your own input** to experiment. Copy the final code cell for submission.

## 1. Problem

### Statement summary

Given a string `s`, return one of its longest contiguous substrings whose characters read identically from left to right and from right to left. The result must be a substring of the original input. If several longest palindromes exist, any of them is acceptable.

### Constraints

- `1 <= len(s) <= 1000`
- `s` contains only digits and English letters.

### Examples

**Example 1**

```text
Input: s = "babad"
Output: "bab"
```

`"bab"` is a length-three palindrome. `"aba"` has the same maximum length and is also accepted.

**Example 2**

```text
Input: s = "cbbd"
Output: "bb"
```

The adjacent `b` characters form the longest palindrome. Its centre lies between two positions.

## 2. Understand the task

Return the **substring itself**, rather than its length or its indices. A substring uses consecutive positions: unlike a subsequence, it cannot skip characters. For example, `"abcda"` does not contain `"aca"` as a substring, even though those letters could be selected in that order.

Compare characters exactly. Uppercase and lowercase letters differ, so `"Aa"` has no length-two palindrome. Digits participate normally. Single characters are always palindromes, and the nonempty input guarantee means an answer exists. Python strings are immutable; both solutions preserve the input.

Every palindrome has one of two centre shapes:

- **Odd length:** a character in the middle, as in `"babad"` → `"bab"`.
- **Even length:** a gap between characters, as in `"cbbd"` → `"bb"`.

Use official example 2, `"cbbd"`, for both main walkthroughs. It has only four positions and makes the often-missed even-centre case explicit. An inclusive palindrome from `left` through `right` has length `right - left + 1`.

In [1]:
# Shared walkthrough input (official example 2).
EXAMPLE_S = "cbbd"
EXAMPLE_EXPECTED = "bb"

## 3. Approach 1 — check every substring

### Idea

Enumerate all nonempty contiguous substrings. A candidate is a palindrome exactly when it equals its reverse. Remember the longest candidate that passes this check.

### Algorithm

1. Initialize `best` as an empty string.
2. Choose every start index `start`.
3. Choose every exclusive end index `end` greater than `start`.
4. Copy `s[start:end]` and compare it with its reverse.
5. If it is a palindrome and strictly longer than `best`, replace `best`.
6. After all candidates, return `best`.

The strict length comparison keeps the existing answer on a tie; the problem allows any longest answer.

### Dry run

`s = "cbbd"`. Each row groups all checks at one start position. This implementation checks every candidate, including candidates shorter than the best already found.

| Start | Candidates in order | Palindromes found | Best after row |
|---:|---|---|---|
| 0 | `c`, `cb`, `cbb`, `cbbd` | `c` | `c` |
| 1 | `b`, `bb`, `bbd` | `b`, `bb` | `bb` |
| 2 | `b`, `bd` | `b` | `bb` |
| 3 | `d` | `d` | `bb` |

After 10 substring checks, return `"bb"`. In row 1, the single `b` ties the current length and does not replace `c`; `bb` is longer and does replace it.

### Why it works

Every nonempty substring is uniquely identified by its start and exclusive end positions. The loops visit every such pair, so no eligible substring is missed. Comparing with the reversed copy accepts exactly palindromes. After each check, `best` is a longest palindrome among the candidates seen so far. When the loops finish, it is therefore a longest palindrome in the entire input. Nonempty input guarantees at least a single-character candidate passes.

### Complexity

Let `n = len(s)`.

- **Worst-case time:** O(n³). There are O(n²) substrings. Copying a substring and reversing it takes O(length), and comparison costs up to O(length). The sum of all candidate lengths is n(n + 1)(n + 2)/6.
- **Auxiliary space:** O(n), including the temporary candidate and reversed strings and the best string retained during the scan. No list of all candidates is stored.

The baseline is useful for small examples, but performs unnecessary work at the 1000-character limit.

### Implementation

In [2]:
class Solution:
    def longestPalindrome(self, s: str) -> str:
        best = ""
        for start in range(len(s)):
            for end in range(start + 1, len(s) + 1):
                candidate = s[start:end]
                if candidate == candidate[::-1] and len(candidate) > len(best):
                    best = candidate
        return best


BruteForceSolution = Solution
print(BruteForceSolution().longestPalindrome(EXAMPLE_S))

bb


In [3]:
# Teaching trace, separate from the solution implementation.
def trace_brute_force(s):
    best = ""
    checks = 0
    for start in range(len(s)):
        for end in range(start + 1, len(s) + 1):
            candidate = s[start:end]
            palindrome = candidate == candidate[::-1]
            if palindrome and len(candidate) > len(best):
                best = candidate
            checks += 1
            print(f"Check {checks}: s[{start}:{end}]={candidate!r}, palindrome={palindrome}, best={best!r}")
    print(f"Return {best!r}; substring checks = {checks}")
    return best

assert trace_brute_force(EXAMPLE_S) == EXAMPLE_EXPECTED

Check 1: s[0:1]='c', palindrome=True, best='c'
Check 2: s[0:2]='cb', palindrome=False, best='c'
Check 3: s[0:3]='cbb', palindrome=False, best='c'
Check 4: s[0:4]='cbbd', palindrome=False, best='c'
Check 5: s[1:2]='b', palindrome=True, best='c'
Check 6: s[1:3]='bb', palindrome=True, best='bb'
Check 7: s[1:4]='bbd', palindrome=False, best='bb'
Check 8: s[2:3]='b', palindrome=True, best='bb'
Check 9: s[2:4]='bd', palindrome=False, best='bb'
Check 10: s[3:4]='d', palindrome=True, best='bb'
Return 'bb'; substring checks = 10


## 4. Approach 2 — expand around centres

### From rechecking whole substrings to extending known palindromes

The baseline checks many overlapping substrings from scratch. A palindrome can instead be built from its centre: if its interior is already symmetric, only the next two outer characters need comparison.

For every index `center`, try `(center, center)` for odd-length palindromes and `(center, center + 1)` for even-length palindromes. Expand while both positions exist and their characters match. Keep only the best start and length; copy the winning substring once, at the end.

### Algorithm

1. Initialize `best_start = best_length = 0`.
2. For each centre index, process the odd centre, then the even centre.
3. While the two boundary characters match and are in range, move `left` one step left and `right` one step right.
4. When expansion stops, the last valid palindrome occupies `left + 1` through `right - 1`. Return its start `left + 1` and length `right - left - 1` from the helper.
5. Replace the best boundaries only if the new length is larger.
6. Return the final slice `s[best_start:best_start + best_length]`.

No matching pair at an even centre gives length `0`, which cannot replace a nonempty answer. The final even-centre call starts with `right == n`; the bounds check handles it immediately.

### Dry run

`s = "cbbd"`

| Centre index | Odd expansion result | Even expansion result | Best after both |
|---:|---|---|---|
| 0 | `c`; next left index is outside | empty: `c != b` | `c` |
| 1 | `b`; outer `c != b` | `bb`; outer `c != d` | `bb` |
| 2 | `b`; outer `b != d` | empty: `b != d` | `bb` |
| 3 | `d`; next right index is outside | empty: right starts outside | `bb` |

At even centre `(1, 2)`, the `b` pair matches. The pointers move to `(0, 3)`, where `c != d`, so expansion returns start `1` and length `3 - 0 - 1 = 2`. After all eight helper calls, return `"bb"`.

### Why it works

Each successful expansion surrounds an already palindromic interior with two equal characters. This preserves the palindrome property. When a comparison fails, no larger palindrome with that same centre can exist: it would contain the mismatched pair. Reaching an input boundary also prevents further growth. Thus each helper call finds the longest palindrome for its centre shape.

Every nonempty palindrome has either one central character or a central gap. All such centres are visited, so a globally longest palindrome is contained in one of the expansions. The algorithm records the longest result seen and therefore returns a globally longest palindromic substring. Keeping a previous answer on a tie remains valid.

### Complexity

- **Worst-case time:** O(n²). There are `2n` helper calls and each can expand O(n) positions. A string of identical characters realizes quadratic total expansion work. The final answer slice takes O(L), where `L <= n` is its length, within the same overall bound.
- **Auxiliary space:** O(1), excluding the returned string. The helper uses indices and returns a fixed-size tuple; it has no recursion and creates no candidate substrings while expanding. The returned slice itself occupies O(L) space in the usual Python slicing model.

### Implementation

In [4]:
class Solution:
    def longestPalindrome(self, s: str) -> str:
        n = len(s)
        best_start = best_length = 0

        def expand(left: int, right: int) -> tuple[int, int]:
            while left >= 0 and right < n and s[left] == s[right]:
                left -= 1
                right += 1
            return left + 1, right - left - 1

        for center in range(n):
            for left, right in ((center, center), (center, center + 1)):
                start, length = expand(left, right)
                if length > best_length:
                    best_start, best_length = start, length
        return s[best_start:best_start + best_length]


CenterExpansionSolution = Solution
print(CenterExpansionSolution().longestPalindrome(EXAMPLE_S))

bb


In [5]:
def trace_centers(s):
    n = len(s)
    best_start = best_length = 0
    calls = 0
    for center in range(n):
        for kind, left, right in (("odd", center, center), ("even", center, center + 1)):
            calls += 1
            print(f"Call {calls}: {kind} centre ({left}, {right})")
            while left >= 0 and right < n and s[left] == s[right]:
                print(f"  Match ({left}, {right}): {s[left]!r} == {s[right]!r}")
                left -= 1
                right += 1
            if left < 0 or right >= n:
                print(f"  Stop: boundary reached at ({left}, {right})")
            else:
                print(f"  Stop: {s[left]!r} != {s[right]!r} at ({left}, {right})")
            start, length = left + 1, right - left - 1
            if length > best_length:
                best_start, best_length = start, length
            print(f"  Candidate={s[start:start + length]!r}; best={s[best_start:best_start + best_length]!r}")
    result = s[best_start:best_start + best_length]
    print(f"Return {result!r}; centre calls = {calls}")
    return result

assert trace_centers(EXAMPLE_S) == EXAMPLE_EXPECTED

Call 1: odd centre (0, 0)
  Match (0, 0): 'c' == 'c'
  Stop: boundary reached at (-1, 1)
  Candidate='c'; best='c'
Call 2: even centre (0, 1)
  Stop: 'c' != 'b' at (0, 1)
  Candidate=''; best='c'
Call 3: odd centre (1, 1)
  Match (1, 1): 'b' == 'b'
  Stop: 'c' != 'b' at (0, 2)
  Candidate='b'; best='c'
Call 4: even centre (1, 2)
  Match (1, 2): 'b' == 'b'
  Stop: 'c' != 'd' at (0, 3)
  Candidate='bb'; best='bb'
Call 5: odd centre (2, 2)
  Match (2, 2): 'b' == 'b'
  Stop: 'b' != 'd' at (1, 3)
  Candidate='b'; best='bb'
Call 6: even centre (2, 3)
  Stop: 'b' != 'd' at (2, 3)
  Candidate=''; best='bb'
Call 7: odd centre (3, 3)
  Match (3, 3): 'd' == 'd'
  Stop: boundary reached at (2, 4)
  Candidate='d'; best='bb'
Call 8: even centre (3, 4)
  Stop: boundary reached at (3, 4)
  Candidate=''; best='bb'
Return 'bb'; centre calls = 8


## 5. Comparison and common mistakes

| Approach | Worst-case time | Auxiliary space | Main trade-off |
|---|---|---|---|
| Enumerate substrings and reverse each | O(n³) | O(n) | Direct, but copies and rechecks overlapping candidates |
| Expand around odd and even centres | O(n²) | O(1), excluding output | Reuses symmetry and stores only winning boundaries |

Choose centre expansion for the final submission: it is concise, handles both palindrome shapes, and avoids a quadratic table. Dynamic programming can also reach O(n²) time with a table of substring results; Manacher's algorithm achieves O(n) time using palindrome radii and mirror information. The two implementations here focus on the direct baseline and centre expansion; the official topic list includes those other techniques as well.

Common mistakes:

- Checking only odd centres and missing answers such as `"bb"` or `"abba"`.
- Returning the length instead of the substring.
- Allowing skipped characters, thereby solving a subsequence problem.
- Forgetting that pointers stop one step outside the last valid palindrome.
- Using `right - left + 1` after expansion stops; the correct length then is `right - left - 1`.
- Constructing a slice inside every expansion step while still claiming constant auxiliary space and the same implementation costs.
- Lowercasing the input or removing digits, which changes the problem.
- Rejecting an alternative longest answer such as `"aba"` for `"babad"`.

Printed teaching traces use slices for readability and have their own overhead. Manual validation and parsing are also separate from submission complexity. The solution methods assume the official input contract; their incidental empty-string result is not a promise that an empty input satisfies the platform constraints.

## 6. Test the solutions

The first two cases reproduce all official examples. Supplementary inputs cover one character, unequal neighbours, a full even palindrome, repeated digits, case sensitivity, a longest palindrome near the end, and ties. Check that every answer is contiguous, palindromic, and of the known optimum length rather than requiring a particular tied answer.

Both implementations run on the short cases. Exercise the centre algorithm at the 1000-character limit separately; avoid making the cubic baseline process that case during Run All.

In [6]:
def check_answer(s, answer, expected_length):
    assert isinstance(answer, str)
    assert answer in s, "The answer must be a contiguous substring."
    assert answer == answer[::-1], "The answer must be a palindrome."
    assert len(answer) == expected_length, (s, answer, expected_length)

TEST_CASES = [
    ("Problem example 1", "babad", 3),
    ("Problem example 2", "cbbd", 2),
    ("Supplementary: one character", "a", 1),
    ("Supplementary: no matching neighbours", "ac", 1),
    ("Supplementary: full even palindrome", "abba", 4),
    ("Supplementary: identical digits", "1111", 4),
    ("Supplementary: case matters", "Aa", 1),
    ("Supplementary: digit palindrome", "x12321y", 5),
    ("Supplementary: best near end", "abcdd", 2),
    ("Supplementary: tied best answers", "abacdc", 3),
    ("Supplementary: full odd palindrome", "racecar", 7),
]
for name, s, expected_length in TEST_CASES:
    answers = []
    for implementation in (BruteForceSolution, CenterExpansionSolution):
        answer = implementation().longestPalindrome(s)
        check_answer(s, answer, expected_length)
        answers.append(answer)
    print(f"PASS | {name}: {answers}")

boundary_s = "a" * 1000
check_answer(boundary_s, CenterExpansionSolution().longestPalindrome(boundary_s), 1000)
print("PASS | Supplementary: maximum-length repeated string (centre expansion)")

PASS | Problem example 1: ['bab', 'bab']
PASS | Problem example 2: ['bb', 'bb']
PASS | Supplementary: one character: ['a', 'a']
PASS | Supplementary: no matching neighbours: ['a', 'a']
PASS | Supplementary: full even palindrome: ['abba', 'abba']
PASS | Supplementary: identical digits: ['1111', '1111']
PASS | Supplementary: case matters: ['A', 'A']
PASS | Supplementary: digit palindrome: ['12321', '12321']
PASS | Supplementary: best near end: ['dd', 'dd']
PASS | Supplementary: tied best answers: ['aba', 'aba']
PASS | Supplementary: full odd palindrome: ['racecar', 'racecar']
PASS | Supplementary: maximum-length repeated string (centre expansion)


## 7. Try your own input

Edit `s_text` and optionally `approach`, then rerun the next cell. `s_text` contains a JSON string including its double quotes, for example `'"babad"'`. Choose `"center"` for normal use or `"brute_force"` for short demonstrations.

The harness validates a length of 1 to 1000 and permits only ASCII English letters and digits, without changing case. A harness-only limit of 200 characters for brute force avoids accidental long cubic runs; use `"center"` above that limit. JSON parsing and the O(n) validation pass are separate from the platform algorithm. Invalid input produces a clear message; no prompt blocks **Run All**.

In [7]:
import json

def run_manual(s_text, approach="center"):
    try:
        s = json.loads(s_text)
        if not isinstance(s, str) or not 1 <= len(s) <= 1000:
            raise ValueError("Input must be a JSON string with 1 to 1000 characters.")
        if not s.isascii() or not s.isalnum():
            raise ValueError("Use only English letters A-Z, a-z, and digits 0-9.")
        implementations = {"brute_force": BruteForceSolution, "center": CenterExpansionSolution}
        if approach not in implementations:
            raise ValueError("approach must be 'brute_force' or 'center'.")
        if approach == "brute_force" and len(s) > 200:
            raise ValueError("For more than 200 characters, select 'center'.")
        answer = implementations[approach]().longestPalindrome(s)
        preview = repr(answer) if len(answer) <= 60 else repr(answer[:57] + "...")
        print(f"{approach}: answer={preview}; length={len(answer)}")
        return answer
    except (ValueError, TypeError) as error:
        print(f"Input error: {error}")
        return None

# EDIT THESE VALUES, then run this cell again.
s_text = '"babad"'
approach = "center"  # Or "brute_force" for short inputs.
manual_result = run_manual(s_text, approach)

center: answer='bab'; length=3


In [8]:
assert run_manual('"cbbd"', "brute_force") == "bb"
assert run_manual('"a"') == "a"
assert run_manual('""') is None
assert run_manual('["a"]') is None
assert run_manual('true') is None
assert run_manual('not JSON') is None
assert run_manual('"a a"') is None
assert run_manual('"a!a"') is None
assert run_manual(json.dumps("é")) is None
assert run_manual(json.dumps("a" * 1001)) is None
assert run_manual('"aba"', "unknown") is None
assert run_manual(json.dumps("a" * 201), "brute_force") is None
print("Manual-input harness checks passed.")

brute_force: answer='bb'; length=2
center: answer='a'; length=1
Input error: Input must be a JSON string with 1 to 1000 characters.
Input error: Input must be a JSON string with 1 to 1000 characters.
Input error: Input must be a JSON string with 1 to 1000 characters.
Input error: Expecting value: line 1 column 1 (char 0)
Input error: Use only English letters A-Z, a-z, and digits 0-9.
Input error: Use only English letters A-Z, a-z, and digits 0-9.
Input error: Use only English letters A-Z, a-z, and digits 0-9.
Input error: Input must be a JSON string with 1 to 1000 characters.
Input error: approach must be 'brute_force' or 'center'.
Input error: For more than 200 characters, select 'center'.
Manual-input harness checks passed.


## 8. Final submission

Copy the next cell into LeetCode's Python 3 editor. It uses the verified `Solution.longestPalindrome(self, s: str) -> str` interface, expands around both centre shapes, and creates only the final answer slice. No imports or notebook helpers are needed.

In [9]:
class Solution:
    def longestPalindrome(self, s: str) -> str:
        n = len(s)
        best_start = best_length = 0

        def expand(left: int, right: int) -> tuple[int, int]:
            while left >= 0 and right < n and s[left] == s[right]:
                left -= 1
                right += 1
            return left + 1, right - left - 1

        for center in range(n):
            for left, right in ((center, center), (center, center + 1)):
                start, length = expand(left, right)
                if length > best_length:
                    best_start, best_length = start, length
        return s[best_start:best_start + best_length]